# AoC 2024 Day 14 — Restroom Redoubt

**Spark — closed-form modular arithmetic + conditional aggregation**

Puzzle: <https://adventofcode.com/2024/day/14>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A list of robots on a wrapping grid — the real floor is **101 tiles wide and 103 tall** (the published example uses 11×7). Each line gives a starting position `p=x,y` and a velocity `v=x,y` in tiles per second, where `x` counts from the left and `y` from the top.

Robots move in straight lines, teleport to the opposite edge when they run off one side, and ignore each other completely — several can share a tile.

- **Part 1** — run 100 seconds, split the floor into four quadrants, and count the robots in each. Robots sitting exactly on the middle row or middle column count for no quadrant. The **safety factor** is the four counts multiplied together.

## The approach

The robots **do not interact** — the puzzle says so outright, they pass over and under each other. That single sentence removes every reason to simulate.

Without interaction, position at time *t* is closed form:

```
x = (px + t·vx) mod width
y = (py + t·vy) mod height
```

No loop over 100 seconds, no intermediate state. The reference implementation steps one second at a time and does 100× the work for the same answer.

So the whole day is **one projection plus one aggregate**. Quadrant counting is not even a `groupBy` — there are exactly four buckets known at plan time, so four `sum(predicate.cast("long"))` expressions in a single `agg` scan the data once and return one row. A `groupBy` on a derived quadrant column would give the same answer through a shuffle; this way there is none.

The robots exactly on a middle row or column belong to no quadrant. That falls out for free: the predicates are strict (`< mid` and `> mid`, never `<=`), so a robot on the centre line satisfies neither and is counted nowhere.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day14

spark = get_spark('aoc-2024-day14')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

The example runs at a different scale than the real input, so `height=7`, `width=11` are passed explicitly rather than hardcoded.

In [ ]:
EXAMPLE = 'p=0,4 v=3,-3\np=6,3 v=-1,-3\np=10,3 v=-1,2\np=2,0 v=2,-1\np=0,0 v=1,3\np=3,0 v=-2,-2\np=7,6 v=-1,-3\np=3,0 v=-1,-2\np=9,3 v=2,3\np=7,3 v=-1,2\np=2,4 v=2,-3\np=9,5 v=-3,-3\n'

print('part 1:', day14.part1(spark, EXAMPLE, height=7, width=11), '(expected 12)')

### The double mod, and who gets discarded

Compare `naive_x` with `x`. Then read the quadrant table: the `-` bucket is the robots sitting on a midline, which the safety factor throws away.

In [ ]:
from pyspark.sql import functions as F

# The example uses an 11x7 floor, not the real 101x103.
width, height = 11, 7
robots = day14.parse(spark, EXAMPLE)

raw_x = (F.col('px') + 100 * F.col('vx')) % width
raw_y = (F.col('py') + 100 * F.col('vy')) % height

# Spark's % keeps the dividend's sign -- watch naive_x go negative.
moved = robots.select(
    'px',
    'py',
    'vx',
    'vy',
    raw_x.alias('naive_x'),
    raw_y.alias('naive_y'),
    ((raw_x + width) % width).alias('x'),
    ((raw_y + height) % height).alias('y'),
)
moved.show()

mid_x, mid_y = width // 2, height // 2
print('midlines: x =', mid_x, ' y =', mid_y)
moved.select(
    'x',
    'y',
    F.when(F.col("x") == mid_x, F.lit("-"))
    .when(F.col("y") == mid_y, F.lit("-"))
    .otherwise(
        F.concat(
            F.when(F.col("y") < mid_y, F.lit("t")).otherwise(F.lit("b")),
            F.when(F.col("x") < mid_x, F.lit("l")).otherwise(F.lit("r")),
        )
    ).alias("quadrant"),
).groupBy("quadrant").count().orderBy("quadrant").show()

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 14)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day14.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day14 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- **The double mod is the bug this day is famous for.** Spark's `%` follows SQL and keeps the sign of the **dividend**, so `-7 % 101` is `-7`, not `94`. The input has negative velocities, `px + 100·vx` goes deeply negative, and a single `%` leaves it there. `((v % n) + n) % n` is the fix — the second mod is not redundant, it is what normalises the negative branch.
- The regex is `-?\d+`. Drop the `-?` and the minus signs are silently lost, every velocity becomes positive, and the answer is wrong without any error.
- Grid size is a **parameter**, not a constant, because the published example runs on 11×7 while the real input is 101×103. Running the example at the default 101×103 gives a wrong answer with no complaint.
- Both dimensions are **odd**, so `width // 2` is a genuine middle column with an equal number of columns either side. The strict `<` / `>` predicates depend on this; on an even grid there is no middle line to exclude and the split would need rethinking.
- Quadrant sums are `.cast("long")` on booleans — `true` → 1, `false` → 0. Under ANSI mode boolean-to-long is still a legal cast.
- `int(quadrants[name] or 0)` handles an empty quadrant, where `sum` returns null rather than 0. One null would otherwise poison the whole product.
- Positions are `INT`. `px + 100·vx` stays far inside 32 bits at t=100; a much larger *t* would not.